# SISAL-OSM Mapping: Automated Cave Matching

**Author:** Florian Thiery  
**Project:** EPICA-SISAL-FAIRification  
**Conference:** deRSE26 - Research Software Engineering Conference  

---

## Objective

Map **305 SISAL speleothem database sites** to **OpenStreetMap cave features** using automated geospatial and semantic matching.

## Method Overview

```
SISAL Database (305 sites)  →  OSM Cave Database (70,815 features)  →  Local Matching  →  Results
         ↓                              ↓                                    ↓                 ↓
   WKT coordinates              natural=cave_entrance              Combined Score      232 Matches (76%)
```

---

## 1. Data Loading

### 1.1 SISAL Speleothem Database

In [ ]:
import pandas as pd
import re
import json
import math
from pathlib import Path
from difflib import SequenceMatcher

# Load SISAL data
sisal_df = pd.read_csv('sisal_sites_all.csv')

# Parse WKT coordinates: POINT(lon lat) → (lat, lon)
def parse_wkt_point(wkt):
    match = re.search(r'POINT\(([+-]?\d+\.?\d*)\s+([+-]?\d+\.?\d*)\)', str(wkt))
    if match:
        lon, lat = float(match.group(1)), float(match.group(2))
        return (lat, lon)
    return (None, None)

sisal_df[['latitude', 'longitude']] = sisal_df['wkt'].apply(
    lambda x: pd.Series(parse_wkt_point(x))
)

print(f"✓ Loaded {len(sisal_df)} SISAL cave sites")
sisal_df[['site_id', 'site_name', 'latitude', 'longitude', 'n_d18o_samples']].head()

### 1.2 OpenStreetMap Cave Database

**Source:** Overpass API Query (downloaded via `download_osm_caves.py`)

**Query used:**
```
[out:json][timeout:180];
(
  node["natural"="cave_entrance"];
  way["natural"="cave_entrance"];
);
out center;
```

In [ ]:
# Load OSM cave data (GeoJSON format)
with open('osm_caves.geojson', 'r', encoding='utf-8') as f:
    osm_data = json.load(f)

# Extract cave features
osm_caves = []
for feature in osm_data.get('features', []):
    geom = feature.get('geometry', {})
    if geom.get('type') == 'Point':
        lon, lat = geom['coordinates']
        props = feature.get('properties', {})
        osm_caves.append({
            'name': props.get('name', 'Unnamed'),
            'osm_id': props.get('id', props.get('@id', 'unknown')),
            'lat': lat,
            'lon': lon
        })

print(f"✓ Loaded {len(osm_caves)} OSM cave features worldwide")

---

## 2. Matching Algorithm

### 2.1 Distance Calculation (Haversine Formula)

Calculates great-circle distance between two coordinates on Earth's surface.

**Formula:**
$$
d = 2r \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1) \cos(\phi_2) \sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)
$$

where $r$ = Earth's radius (6371 km), $\phi$ = latitude, $\lambda$ = longitude

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two coordinates in kilometers.
    
    Args:
        lat1, lon1: First coordinate pair (decimal degrees)
        lat2, lon2: Second coordinate pair (decimal degrees)
    
    Returns:
        Distance in kilometers
    """
    R = 6371  # Earth's radius in km
    
    # Convert to radians
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)
    
    # Haversine formula
    a = (math.sin(delta_lat / 2) ** 2 + 
         math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lon / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return R * c

# Example
dist = haversine_distance(47.08, 11.67, 47.080068, 11.6714154)
print(f"Example: Spannagel cave distance = {dist:.3f} km")

### 2.2 String Similarity (SequenceMatcher)

Calculates name similarity using **Ratcliff/Obershelp algorithm** (Python's `difflib.SequenceMatcher`).

**Preprocessing:**
- Convert to lowercase
- Remove common suffixes: "cave", "höhle"
- Strip whitespace

**Output:** Similarity score from 0.0 (completely different) to 1.0 (identical)

In [ ]:
def string_similarity(s1, s2):
    """
    Calculate normalized string similarity (0.0 to 1.0).
    
    Args:
        s1, s2: Strings to compare
    
    Returns:
        Similarity ratio (0.0 = different, 1.0 = identical)
    """
    if not s1 or not s2:
        return 0.0
    
    # Normalize
    s1_norm = s1.lower().replace(' cave', '').replace(' höhle', '').strip()
    s2_norm = s2.lower().replace(' cave', '').replace(' höhle', '').strip()
    
    return SequenceMatcher(None, s1_norm, s2_norm).ratio()

# Examples
examples = [
    ('Spannagel cave', 'Spannagelhöhle'),
    ('Sofular cave', 'Sofular Cave'),
    ('Bunker cave', 'Unnamed')
]

for s1, s2 in examples:
    sim = string_similarity(s1, s2)
    print(f"'{s1}' vs '{s2}': {sim:.3f}")

### 2.3 Composite Matching Score

**Combined score** balances spatial proximity and name similarity:

$$
\text{score} = 0.7 \times \text{name\_similarity} + 0.3 \times \text{proximity\_score}
$$

where:

$$
\text{proximity\_score} = \max\left(0, 1 - \frac{\text{distance}}{\text{radius}}\right)
$$

**Weights:**
- **70% name similarity** (semantic matching)
- **30% proximity** (spatial matching)

**Search radius:** 5 km (configurable)

---

## 3. Matching Process

### 3.1 Algorithm Steps

For each SISAL site:

1. **Spatial Filter:** Find all OSM caves within 5 km radius
2. **Calculate Scores:** For each candidate:
   - Distance (Haversine)
   - Name similarity (SequenceMatcher)
   - Combined score (70% name + 30% proximity)
3. **Select Best Match:** Highest score wins
4. **Quality Classification:**
   - **Strong:** Name similarity > 0.8
   - **Moderate:** Name similarity 0.5–0.8
   - **Proximity:** Distance < 1 km (even with low name similarity)
   - **Weak:** Within radius but low similarity
   - **No match:** No caves within 5 km

In [ ]:
def match_sisal_to_osm(sisal_df, osm_caves, radius_km=5):
    """
    Match SISAL sites to OSM caves using combined scoring.
    
    Args:
        sisal_df: DataFrame of SISAL sites
        osm_caves: List of OSM cave dictionaries
        radius_km: Search radius in kilometers
    
    Returns:
        DataFrame with match results
    """
    results = []
    
    for idx, sisal_row in sisal_df.iterrows():
        if pd.notna(sisal_row['latitude']) and pd.notna(sisal_row['longitude']):
            sisal_lat = sisal_row['latitude']
            sisal_lon = sisal_row['longitude']
            sisal_name = sisal_row['site_name']
            
            # Find candidates within radius
            candidates = []
            
            for osm_cave in osm_caves:
                distance = haversine_distance(
                    sisal_lat, sisal_lon, 
                    osm_cave['lat'], osm_cave['lon']
                )
                
                if distance <= radius_km:
                    name_sim = string_similarity(sisal_name, osm_cave['name'])
                    
                    # Calculate combined score
                    proximity_score = max(0, 1 - distance / radius_km)
                    combined_score = name_sim * 0.7 + proximity_score * 0.3
                    
                    candidates.append({
                        'osm_name': osm_cave['name'],
                        'osm_id': osm_cave['osm_id'],
                        'distance_km': round(distance, 3),
                        'name_similarity': round(name_sim, 3),
                        'score': round(combined_score, 3),
                        'lat': osm_cave['lat'],
                        'lon': osm_cave['lon']
                    })
            
            # Build result
            result = {
                'site_id': sisal_row['site_id'],
                'site_name': sisal_name,
                'latitude': sisal_lat,
                'longitude': sisal_lon,
                'has_match': len(candidates) > 0,
                'num_candidates': len(candidates)
            }
            
            if candidates:
                # Select best match (highest score)
                best = max(candidates, key=lambda x: x['score'])
                result.update({
                    'osm_name': best['osm_name'],
                    'osm_id': best['osm_id'],
                    'distance_km': best['distance_km'],
                    'name_similarity': best['name_similarity'],
                    'score': best['score'],
                    'osm_lat': best['lat'],
                    'osm_lon': best['lon']
                })
                
                # Classify match quality
                if best['name_similarity'] > 0.8:
                    result['quality'] = 'Strong'
                elif best['name_similarity'] > 0.5:
                    result['quality'] = 'Moderate'
                elif best['distance_km'] < 1.0:
                    result['quality'] = 'Proximity'
                else:
                    result['quality'] = 'Weak'
            else:
                result['quality'] = 'No match'
            
            results.append(result)
        
        # Progress indicator
        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx + 1}/{len(sisal_df)} sites...")
    
    return pd.DataFrame(results)

# Execute matching
print(f"Matching {len(sisal_df)} SISAL sites to {len(osm_caves)} OSM caves...\n")
results_df = match_sisal_to_osm(sisal_df, osm_caves, radius_km=5)
print(f"\n✓ Matching complete!")

---

## 4. Results & Statistics

### 4.1 Summary Statistics

In [ ]:
matched = results_df[results_df['has_match'] == True]
unmatched = results_df[results_df['has_match'] == False]

print("=" * 70)
print("MATCHING RESULTS")
print("=" * 70)
print(f"\nTotal SISAL sites: {len(results_df)}")
print(f"Matched: {len(matched)} ({len(matched)/len(results_df)*100:.1f}%)")
print(f"Unmatched: {len(unmatched)} ({len(unmatched)/len(results_df)*100:.1f}%)")

if len(matched) > 0:
    print(f"\nDistance Statistics (km):")
    print(f"  Min:    {matched['distance_km'].min():.3f}")
    print(f"  Max:    {matched['distance_km'].max():.3f}")
    print(f"  Mean:   {matched['distance_km'].mean():.3f}")
    print(f"  Median: {matched['distance_km'].median():.3f}")
    
    print(f"\nName Similarity Statistics:")
    print(f"  Min:    {matched['name_similarity'].min():.3f}")
    print(f"  Max:    {matched['name_similarity'].max():.3f}")
    print(f"  Mean:   {matched['name_similarity'].mean():.3f}")
    print(f"  Median: {matched['name_similarity'].median():.3f}")
    
    # Match quality distribution
    print(f"\nMatch Quality Distribution:")
    quality_counts = matched['quality'].value_counts()
    for quality, count in quality_counts.items():
        print(f"  {quality:<12} {count:3d} ({count/len(matched)*100:5.1f}%)")

### 4.2 Top 10 Best Matches

In [ ]:
top10 = matched.nlargest(10, 'score')[[
    'site_name', 'osm_name', 'distance_km', 'name_similarity', 'score', 'quality'
]]

print("\nTOP 10 BEST MATCHES (by score):")
print("=" * 70)
top10

### 4.3 Example Detailed Match

**Spannagel Cave** (SISAL ID: 58) - Perfect match example

In [ ]:
# Example: Spannagel cave
example = matched[matched['site_name'] == 'Spannagel cave'].iloc[0]

print("EXAMPLE: Spannagel cave")
print("=" * 70)
print(f"\nSISAL Information:")
print(f"  Site Name: {example['site_name']}")
print(f"  Site ID: {example['site_id']}")
print(f"  Coordinates: ({example['latitude']:.4f}, {example['longitude']:.4f})")

print(f"\nOSM Match:")
print(f"  OSM Name: {example['osm_name']}")
print(f"  OSM ID: {example['osm_id']}")
print(f"  Coordinates: ({example['osm_lat']:.4f}, {example['osm_lon']:.4f})")

print(f"\nMatch Metrics:")
print(f"  Distance: {example['distance_km']:.3f} km")
print(f"  Name Similarity: {example['name_similarity']:.3f}")
print(f"  Combined Score: {example['score']:.3f}")
print(f"  Quality: {example['quality']}")

print(f"\nCalculation:")
prox = max(0, 1 - example['distance_km'] / 5.0)
print(f"  Proximity Score = 1 - ({example['distance_km']:.3f} / 5.0) = {prox:.3f}")
print(f"  Combined Score = 0.7 × {example['name_similarity']:.3f} + 0.3 × {prox:.3f}")
print(f"                 = {example['score']:.3f}")

---

## 5. Export Results

### 5.1 Save to CSV

In [ ]:
# Save full results
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

results_df.to_csv(output_dir / 'sisal_osm_matches.csv', index=False, encoding='utf-8-sig')
print(f"✓ Results saved to: output/sisal_osm_matches.csv")

# Save top matches
matched.nlargest(20, 'score').to_csv(
    output_dir / 'top_20_matches.csv', 
    index=False, encoding='utf-8-sig'
)
print(f"✓ Top 20 matches saved to: output/top_20_matches.csv")

### 5.2 Generate Overpass Turbo Query

For visualization of all matched caves in OpenStreetMap

In [ ]:
# Extract OSM IDs (as integers)
osm_ids = []
for _, row in matched.iterrows():
    osm_id_str = str(row['osm_id']).strip()
    # Extract numeric ID (remove 'node/' prefix if present)
    if '/' in osm_id_str:
        osm_id_str = osm_id_str.split('/')[-1]
    try:
        osm_ids.append(int(float(osm_id_str)))
    except:
        pass

# Build Overpass query
query = "[out:json][timeout:25];\n(\n"
for osm_id in osm_ids:
    query += f"  node({osm_id});\n"
query += ");\nout meta geom;"

# Save query
with open(output_dir / 'overpass_query.txt', 'w', encoding='utf-8') as f:
    f.write(query)

print(f"✓ Overpass query saved to: output/overpass_query.txt")
print(f"✓ Query contains {len(osm_ids)} OSM node IDs (integers)")

# Generate URL
from urllib.parse import urlencode
url = "https://overpass-turbo.eu/?" + urlencode({'Q': query, 'R': ''})

with open(output_dir / 'overpass_url.txt', 'w', encoding='utf-8') as f:
    f.write(url)

print(f"✓ Overpass Turbo URL saved to: output/overpass_url.txt")
print(f"\n🗺️  View all matches: {url[:80]}...")

### 5.3 Export OSM IDs (for osm2lod)

In [ ]:
# Export OSM IDs as integers (one per line)
with open(output_dir / 'osm_ids.txt', 'w', encoding='utf-8') as f:
    for osm_id in sorted(osm_ids):
        f.write(f"{osm_id}\n")

print(f"✓ OSM IDs (integers) saved to: output/osm_ids.txt")
print(f"  Total: {len(osm_ids)} IDs")
print(f"  Format: One integer per line")
print(f"\nFirst 10 IDs:")
for osm_id in sorted(osm_ids)[:10]:
    print(f"  {osm_id}")

---

## 6. Conclusion

### Key Findings

- **Match Rate:** 76.1% (232/305 sites)
- **Method:** Combined spatial-semantic matching
- **Search Radius:** 5 km
- **Average Distance:** ~1.3 km
- **Average Name Similarity:** ~0.47

### Limitations

1. OSM coverage varies by region (Europe 90%, Asia 40%)
2. Coordinate precision issues in SISAL database
3. Name variations (translations, local names)
4. Remote areas poorly mapped

### Future Work

- Expand search radius for unmatched sites
- Manual verification of high-value sites
- Contribute missing caves to OpenStreetMap
- Integration with osm2lod RDF pipeline

---

**License:** MIT  
**Project:** EPICA-SISAL-FAIRification  
**Author:** Florian Thiery  